In [ ]:
class Game:

    def observe(self, fun):
        self.callbacks[fun.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def _notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)

    def __init__(self, size=10, nmines=10):
        self.callbacks = {}
        self.mines_grid = []
        self.visibility_grid = []
        self.flag_grid = []
        self.neighbor_mine_counts = []
        self.game_over = False

def create_empty_grid(rows, columns, default_value=False):
    '''
    Erzeugt eine 2D-Liste (rows x columns) mit einem Standardwert.

    Args:
        rows (int): Anzahl Zeilen.
        columns (int): Anzahl Spalten.
        default_value: Startwert in allen Feldern.

    Returns:
        list[list]: Zweidimensionale Liste mit default_value in jedem Feld.
    '''
    return [[default_value for _ in range(columns)] for _ in range(rows)]


def initialize_game(canvas):
    '''
    Initialisiert das Spiel neu:

    - setzt alle globalen Grids zurück
    - setzt den Flaggen-Modus auf False
    - platziert die Minen zufällig
    - berechnet die Nachbarminenanzahlen
    - leert das Log
    - zeichnet das Spielfeld
    '''
    global game_over

    game_over = False

    # Flaggen-Modus bei jedem neuen Spiel deaktivieren
    flag_mode_button.value = False

    mines_grid[:] = create_empty_grid(GRID_SIZE, GRID_SIZE, False)
    visibility_grid[:] = create_empty_grid(GRID_SIZE, GRID_SIZE, False)
    flag_grid[:] = create_empty_grid(GRID_SIZE, GRID_SIZE, False)

    place_mines_randomly()
    neighbor_mine_counts[:] = calculate_neighbor_mine_counts()

    # Log-Bereich einmal löschen bei neuem Spiel
    output_area.clear_output()
    log('Neues Spiel gestartet.')
    log(f'Spielfeld: {GRID_SIZE}x{GRID_SIZE}, Minen: {NUMBER_OF_MINES}')
    log(f'Flaggen-Modus: {'aktiv' if flag_mode_button.value else 'inaktiv'}')

    redraw_board(canvas)


def place_mines_randomly():
    '''
    Platziert NUMBER_OF_MINES Minen zufällig auf dem Spielfeld.

    Es wird garantiert, dass keine Mine doppelt gesetzt wird.
    '''
    if NUMBER_OF_MINES > GRID_SIZE**2:
        raise ValueError('too many mines!')

    placed = 0
    while placed < NUMBER_OF_MINES:
        row = random.randrange(GRID_SIZE)
        col = random.randrange(GRID_SIZE)

        if not mines_grid[row][col]:
            mines_grid[row][col] = True
            placed += 1
    log(f'{NUMBER_OF_MINES} Minen zufällig platziert.')


def calculate_neighbor_mine_counts():
    '''
    Berechnet für jedes Feld die Anzahl benachbarter Minen.

    Returns:
        list[list[int]]: 2D-Liste der Nachbarminenanzahlen.
    '''
    counts = create_empty_grid(GRID_SIZE, GRID_SIZE, 0)

    for row in range(GRID_SIZE):
        for col in range(GRID_SIZE):
            if not mines_grid[row][col]:
                counts[row][col] = count_neighbor_mines(row, col)

    return counts


def count_neighbor_mines(row, col):
    '''
    Zählt Minen in den acht Nachbarfeldern eines Feldes.

    Args:
        row (int): Zeilenindex.
        col (int): Spaltenindex.

    Returns:
        int: Anzahl Minen in der direkten Nachbarschaft.
    '''
    count = 0

    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue

            r = row + dr
            c = col + dc

            if 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE:
                if mines_grid[r][c]:
                    count += 1

    return count


def get_neighbors(row, col):
    '''
    Liefert alle gültigen Nachbarfelder eines Feldes.

    Args:
        row (int): Zeilenindex.
        col (int): Spaltenindex.

    Returns:
        list[tuple[int, int]]: Liste der Nachbarpositionen als (row, col).
    '''
    neighbors = []

    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue

            r = row + dr
            c = col + dc

            if 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE:
                neighbors.append((r, c))

    return neighbors

    
    def new_game(self):
        self.games_played += 1
        self.pos = (4, 4)
        self._notify('new_game', games_played=self.games_played)

    def is_inside(self, pos):
        return (0 <= pos[0] < self.grid_size
                and 0 <= pos[1] < self.grid_size)

    def push_stone(self, dpos):
        dx, dy = dpos
        x, y = self.pos
        npos = (x+dx, y + dy)

        if not self.is_inside(npos):
            self._notify('error',
                         msg=f'cannot push stone from  {self.pos} to {npos}!',
                         )
            return

        self.pos = npos
        self._notify('push_stone', old=(x, y), new=npos)

    def __repr__(self):
        return f'Games played: {self.games_played}, grid size: {self.grid_size},\n'\
               f'pos: {self.pos}'